# Evaluation Visualization

This notebook visualizes the evaluation metrics and qualitative examples from `evaluation_results.json`.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
with open('../evaluation_results.json') as f:
    results = json.load(f)

print(f"Test Loss:       {results['test_loss']:.4f}")
print(f"Test Perplexity: {results['test_perplexity']:.2f}")
print(f"BLEU:            {results['bleu']:.4f}")
print(f"ROUGE-1 F1:      {results['rouge1']:.4f}")
print(f"ROUGE-2 F1:      {results['rouge2']:.4f}")
print(f"ROUGE-L F1:      {results['rougeL']:.4f}")

In [ ]:
# Bar chart of evaluation metrics
metrics = ['BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L']
values = [results['bleu'], results['rouge1'], results['rouge2'], results['rougeL']]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(metrics, values, color=colors, width=0.6)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.2%}', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Score')
ax.set_title('Evaluation Metrics (Test Set)')
ax.set_ylim(0, 0.65)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('evaluation_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compare test perplexity vs validation perplexity at epoch 10
with open('../training_history.json') as f:
    history = json.load(f)

val_ppl_final = np.exp(history['val_loss'][-1])
test_ppl = results['test_perplexity']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Validation (Epoch 10)', 'Test'], [val_ppl_final, test_ppl],
              color=['#FF5722', '#2196F3'], width=0.5)
for bar, val in zip(bars, [val_ppl_final, test_ppl]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val:.2f}', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Perplexity')
ax.set_title('Validation vs Test Perplexity')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Qualitative Examples

Comparing reference docstrings to model predictions on test set examples.

In [ ]:
for i, ex in enumerate(results['examples'], 1):
    print(f"--- Example {i} ---")
    print(f"Code:      {ex['code'][:80]}...")
    print(f"Reference: {ex['reference']}")
    print(f"Predicted: {ex['predicted']}")
    print()

## Cross-Attention Matrix

The attention matrix shows how the decoder (output summary tokens) attends to the encoder (source code tokens) when generating each word. Brighter cells indicate stronger attention — i.e., which code tokens the model "looked at" to produce each summary token.

In [ ]:
import os, sys, math
sys.path.insert(0, os.path.abspath('..'))
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from models.transformer import Transformer
from utils.dataset import PAD_ID, SOS_ID, EOS_ID

# Pick device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# Load model
checkpoint = torch.load('../checkpoints/best_model.pt', map_location=device)
model = Transformer(vocab_size=checkpoint['vocab_size'], d_model=checkpoint['d_model'], pad_id=PAD_ID)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

# Load tokenizer
tokenizer = Tokenizer.from_file('../data/tokenizer.json')
tokenizer.decoder = ByteLevelDecoder()
print("Model and tokenizer loaded.")

In [ ]:
def decode_with_attention(model, tokenizer, code_text, max_len=128):
    """Greedy decode while capturing cross-attention weights from the last decoder layer."""
    # Hook to capture cross-attention weights
    attn_weights = []
    
    def hook_fn(module, input, output):
        # nn.MultiheadAttention returns (attn_output, attn_weights)
        # output[1] has shape (batch, tgt_len, src_len)
        if output[1] is not None:
            attn_weights.append(output[1].detach().cpu())
    
    # Register hook on the last decoder layer's cross-attention (multihead_attn)
    last_decoder_layer = model.transformer.decoder.layers[-1]
    # Temporarily enable need_weights
    original_need_weights = getattr(last_decoder_layer.multihead_attn, '_qkv_same_embed_dim', True)
    handle = last_decoder_layer.multihead_attn.register_forward_hook(hook_fn)
    
    # Tokenize source code
    code_ids = tokenizer.encode(code_text).ids[:max_len]
    code_tensor = torch.tensor([code_ids], dtype=torch.long, device=device)
    
    # Greedy decode step by step, collecting attention at each step
    generated = [SOS_ID]
    step_attentions = []  # one attention vector per generated token
    
    with torch.no_grad():
        for _ in range(max_len):
            attn_weights.clear()
            decoder_tensor = torch.tensor([generated], dtype=torch.long, device=device)
            output = model(code_tensor, decoder_tensor)
            next_token = output[0, -1, :].argmax().item()
            
            # Grab the attention for the last generated position
            if attn_weights:
                # shape: (1, tgt_len, src_len) -> take last tgt position
                step_attn = attn_weights[0][0, -1, :].numpy()  # (src_len,)
                step_attentions.append(step_attn)
            
            generated.append(next_token)
            if next_token == EOS_ID:
                break
    
    handle.remove()
    
    # Build attention matrix: (num_generated_tokens, src_len)
    attn_matrix = np.array(step_attentions)
    
    # Decode tokens to readable strings
    src_tokens = [tokenizer.decode([tid]) for tid in code_ids]
    tgt_tokens = [tokenizer.decode([tid]) for tid in generated[1:]]  # skip SOS
    # Remove EOS from display if present
    if generated[-1] == EOS_ID:
        tgt_tokens = tgt_tokens[:-1]
        attn_matrix = attn_matrix[:-1]
    
    return attn_matrix, src_tokens, tgt_tokens

# Use the print_log example (Example 5) — it's short and the prediction is reasonable
code_text = 'def print_log(text, *colors):\n    """Print a log message to standard error."""\n    sys.stderr.write(sprint("{}: {}".format(script_name, text), *colors) + "\\n")'
attn_matrix, src_tokens, tgt_tokens = decode_with_attention(model, tokenizer, code_text)
print(f"Source tokens: {len(src_tokens)}")
print(f"Target tokens: {len(tgt_tokens)}")
print(f"Predicted: {' '.join(tgt_tokens)}")

In [ ]:
# Plot cross-attention heatmap
fig, ax = plt.subplots(figsize=(max(12, len(src_tokens) * 0.4), max(4, len(tgt_tokens) * 0.6)))

im = ax.imshow(attn_matrix, cmap='YlOrRd', aspect='auto', interpolation='nearest')

# Label axes with token strings
ax.set_xticks(range(len(src_tokens)))
ax.set_xticklabels(src_tokens, rotation=90, fontsize=8, fontfamily='monospace')
ax.set_yticks(range(len(tgt_tokens)))
ax.set_yticklabels(tgt_tokens, fontsize=10)

ax.set_xlabel('Source Code Tokens (Encoder)')
ax.set_ylabel('Generated Summary Tokens (Decoder)')
ax.set_title('Cross-Attention Matrix: print_log')

plt.colorbar(im, ax=ax, label='Attention Weight')
plt.tight_layout()
plt.savefig('attention_matrix.png', dpi=150, bbox_inches='tight')
plt.show()